# Calling an Existing Agent

Shows how to retrieve and call an existing persistent Foundry agent by its ID.
The same agent can serve multiple independent conversations — each gets its own thread.

> **Before running:** create an agent with `01_persistent_agent.ipynb` (skip the cleanup cell),
> note the printed agent ID, and set `AZURE_FOUNDRY_AGENT_ID` accordingly.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import AgentStreamEvent
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]
agent_id = os.environ["AZURE_FOUNDRY_AGENT_ID"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("Client ready.")

In [ ]:
# Retrieve the existing agent by ID — no need to recreate it
agent = client.agents.get_agent(agent_id=agent_id)
print(f"Using agent: {agent.name} ({agent.id})")

In [ ]:
# Helper: start a new conversation thread and stream the agent's response
def call_agent(prompt: str) -> None:
    thread = client.agents.create_thread()
    client.agents.create_message(thread_id=thread.id, role="user", content=prompt)

    with client.agents.create_stream(thread_id=thread.id, agent_id=agent.id) as stream:
        for event_type, event_data, _ in stream:
            if event_type == AgentStreamEvent.THREAD_MESSAGE_DELTA:
                for block in event_data.delta.content:
                    if block.type == "text":
                        print(block.text.value, end="", flush=True)
    print()

In [ ]:
# First conversation
print("=== First Conversation ===\n")
call_agent("Write a poem about the sea.")

In [ ]:
# Second conversation — same agent, brand new thread
print("=== Second Conversation ===\n")
call_agent("Write a poem about the mountains.")